# FlowStitch — Complete Pipeline Codebase & Annotated Source

This notebook provides a **complete, annotated walkthrough** of every module in the
`flowstitch` package, integrating the theoretical justifications from the other notebooks
with the actual production code. This is the ground truth of what is implemented.

---

## Package Architecture

```
flowstitch/
├── core/
│   ├── config.py            ← Centralized configuration (FlowStitchConfig)
│   ├── flux_hooking.py      ← AttnProcessorWrapper + FluxDataCapturer
│   ├── tokenizer_utils.py   ← Sliding-window minimal token finder
│   └── serialization.py     ← Tensor save/load utilities
├── extraction/
│   ├── attention_mask.py    ← extract_attention_mask() + otsu_threshold()
│   ├── spectral_mask.py     ← compute_fiedler_mask() (DiffCut adaptation)
│   ├── tda_mask.py          ← extract_tda_mask() (Ripser H₀ clustering)
│   ├── hybrid_mask.py       ← hybrid_semantic_decomposition()
│   └── energy_mask.py       ← Energy density thresholding
├── stitching/
│   ├── kts.py               ← apply_kts() (Kinetic Trajectory Shaping)
│   ├── ema_smoothing.py     ← AttentionEMA (Look-Back Flows)
│   ├── semantic_processor.py ← SemanticGraftingProcessor + inject/remove
│   └── ode_perturbation.py  ← perform_ode_step() (Euler + KTS)
└── pipelines/
    ├── latent_stitching.py  ← run_latent_stitching() [MAIN ENTRY POINT]
    ├── dataset_generation.py ← generate_dataset() [DB CREATION]
    └── mask_compilation.py  ← compile_mask() [MASK AGGREGATION]
```

---

## Table of Contents

| Section | Module | Key Function |
|---------|--------|-------------|
| **§1** | `core/config.py` | `FlowStitchConfig` dataclass |
| **§2** | `core/flux_hooking.py` | `FluxDataCapturer`, `AttnProcessorWrapper` |
| **§3** | `core/tokenizer_utils.py` | `find_token_indices()` sliding-window |
| **§4** | `extraction/attention_mask.py` | `extract_attention_mask()`, `otsu_threshold()` |
| **§5** | `extraction/spectral_mask.py` | `compute_fiedler_mask()` |
| **§6** | `extraction/tda_mask.py` | `extract_tda_mask()` (Ripser H₀) |
| **§7** | `extraction/hybrid_mask.py` | `hybrid_semantic_decomposition()` |
| **§8** | `stitching/kts.py` | `apply_kts()`, `compute_damping_factor()` |
| **§9** | `stitching/ema_smoothing.py` | `AttentionEMA` |
| **§10** | `stitching/semantic_processor.py` | `SemanticGraftingProcessor` |
| **§11** | `stitching/ode_perturbation.py` | `perform_ode_step()` |
| **§12** | `pipelines/dataset_generation.py` | `generate_dataset()` |
| **§13** | `pipelines/latent_stitching.py` | `run_latent_stitching()` |
| **§14** | End-to-End Pipeline Runner | `process_dataset_sample()` |

---


---
## §1 — `core/config.py`: Centralized Configuration

The `FlowStitchConfig` dataclass is the single source of truth for all pipeline parameters. Every hyperparameter is documented here with its mathematical role.

In [ ]:
# ── EXACT SOURCE: flowstitch/core/config.py ─────────────────────────────────
from dataclasses import dataclass, field
from typing import List, Optional
import torch

@dataclass
class FlowStitchConfig:
    '''Centralized configuration for FlowStitch pipelines.
    
    Mathematical roles:
    - lambda_v0: λ in v_stitch = v_amb + D(t)·λ·M·(v_target - v_amb)
    - injection_strength: α in blended = img*(1-α*A) + protected*(α*A)  
    - t_cutoff: τ_c in D(t) = exp(-γ·max(0, t-τ_c))
    - gamma_kts: γ in D(t) = exp(-γ·max(0, t-τ_c))
    - ema_decay: γ_EMA in ā_t = γ·a_t + (1-γ)·ā_{t-1}
    '''
    # Model parameters
    model_id: str = "black-forest-labs/FLUX.1-schnell"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    dtype: torch.dtype = torch.bfloat16
    
    # Dataset generation parameters
    output_root: str = "data/dataset_v1"
    seed: int = 42
    steps: int = 4
    target_layers: List[int] = field(default_factory=lambda: [0, 10])
    
    # Stitching parameters
    ambient_prompt: str = "a crystal clear lake"    # z_ambient (background ODE)
    lambda_v0: float = 1.0        # injection strength λ ∈ [0, 1]
    injection_strength: float = 0.85  # α for semantic grafting
    t_cutoff: float = 0.8         # KTS damping cutoff τ_c
    gamma_kts: float = 5.0        # KTS decay rate γ
    ema_decay: float = 0.3        # EMA temporal smoothing γ_EMA
    use_ema: bool = True
    stitching_mode: str = "dual"  # "mosaico" | "dual" | "full"
    
    def to_dict(self):
        return {k: v for k, v in self.__dict__.items() if not k.startswith('_')}

# Demonstrate config
config = FlowStitchConfig()
print("Default FlowStitchConfig:")
for k, v in config.to_dict().items():
    print(f"  {k:25s} = {v}")


---
## §2 — `core/flux_hooking.py`: Data Extraction via Attention Hooks

The `FluxDataCapturer` intercepts the FLUX transformer's forward pass to extract $x_0$, $v_0$, and cross-attention maps $\{A^{(l)}\}$ without modifying the generation. It operates as a Python context manager.

**Mathematical significance:** The cross-attention weights $A^{(l)}_{ij}$ between image tokens $i$ and text tokens $j$ at layer $l$ are:
$$A^{(l)}_{ij} = \text{softmax}\left(\frac{Q^{(l)}_i \cdot K^{(l)}_j}{\sqrt{d}}\right)$$
where Q is the image query and K is the text key. These are the semantic localization scores.

In [ ]:
import inspect, sys, os
sys.path.insert(0, '/Users/tella/Workspace/FlowStitch')

# Show the source code directly
with open('/Users/tella/Workspace/FlowStitch/flowstitch/core/flux_hooking.py') as f:
    source = f.read()

print(source)
print("\n" + "─"*60)
print("KEY DESIGN DECISIONS:")
print('''
1. AttnProcessorWrapper intercepts BEFORE calling original_processor
   → captures Q, K projections in no_grad() context (VRAM protection)
   → extracts cross-attention slice: attn_weights[:, :, n_text:, :n_text]
     meaning image→text attention (spatial tokens attending to text tokens)

2. FluxDataCapturer._transformer_hook captures x0 and v0 at step=0:
   → x0 = hidden_states at t=0 (initial latent noise z_0)
   → v0 = transformer output at t=0 (initial velocity prediction)
   → x_pred = x0 + v0 (predicted clean image, stored for reference)

3. Context manager interface:
   with FluxDataCapturer(transformer, layers=[0, 10]) as capturer:
       pipe(prompt)  # normal inference — hooks capture automatically
   # capturer.x0, capturer.v0, capturer.attn_maps are populated
''')


---
## §3 — `core/tokenizer_utils.py`: Sliding Window Token Search

T5-XXL tokenizes words into sub-tokens (e.g., 'cube' → ['▁cu', 'be'] or ['▁cube']). The `find_token_indices()` function uses a **minimal sliding window** to find the exact, shortest sequence of token IDs that reconstruct the target word after decoding.

In [ ]:
with open('/Users/tella/Workspace/FlowStitch/flowstitch/core/tokenizer_utils.py') as f:
    source = f.read()
print(source)

print("\n" + "─"*60)
print('''ALGORITHM COMPLEXITY:
- Outer loop: O(N) where N = number of tokens in prompt
- Inner loop: O(N) 
- Total: O(N²) in worst case — acceptable for N ≤ 512 (T5 context length)

WHY THIS IS NEEDED (vs simple substring search):
T5 tokenization is byte-pair encoding (BPE), meaning:
- "a red cube"  → ['▁a', '▁red', '▁cu', 'be'] (cube = 2 tokens)
- "blue sphere" → ['▁blue', '▁sphere'] (sphere = 1 token)
- Fragmentation is prompt-dependent and cannot be predicted without the tokenizer

The sliding window finds the MINIMAL span [i, j) such that decode(tokens[i:j]) contains
the target word — preventing false positives like matching 'ub' inside 'cube'.

REAL EXECUTION OUTPUT (from 07bis Cell 3):
  Ricostruzione semantica trovata: 'cube' → Indici: [3, 4]
  (for prompt "a red cube and a blue sphere")
''')


---
## §4 — `extraction/attention_mask.py`: Semantic Localization

Extracts the per-token attention heatmap for the target object and applies Otsu thresholding to binarize it.

In [ ]:
with open('/Users/tella/Workspace/FlowStitch/flowstitch/extraction/attention_mask.py') as f:
    source = f.read()
print(source)

print("\n" + "─"*60)
import numpy as np
import matplotlib.pyplot as plt

print('''
MATHEMATICAL PIPELINE:
  Input: layer_10_attn shape [1, heads, 4096, text_seq_len]
    → Average across target token indices: attn_map_k = mean over k∈token_indices(attn[:,  :, :, k])
    → Sum across target tokens: attn_map = sum_k attn_map_k
    → Clamp and normalize to [0, 1]
    → Otsu threshold to binary mask
  
  Output: [1, 4096, 1] binary mask (FLUX latent sequence order)

CRITICAL NOTE: 
  normalize=True is essential! Unnormalized cross-attention values vary across
  layers and prompts, making Otsu's threshold unstable. Normalization ensures
  Otsu operates on a consistent [0, 1] distribution.
  (This was the 'Otsu collapse' failure documented in notebook 02.)
''')

# Otsu threshold demonstration
np.random.seed(5)
n = 200
vals_norm = np.concatenate([np.random.beta(1.5, 8, 150), np.random.beta(6, 2, 50) * 0.6 + 0.4])
vals_unnorm = vals_norm * 0.001  # simulate unnormalized (very small values)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('§4 — Otsu Normalized vs Unnormalized', fontweight='bold')
for ax, vals, label, color in zip(
    axes,
    [vals_norm, vals_unnorm],
    ['Normalized [0,1]\n(stable Otsu)', 'Unnormalized (×0.001)\n(Otsu collapses)'],
    ['steelblue', 'red']
):
    bins = 50
    hist, edges = np.histogram(vals, bins=bins)
    ax.bar(edges[:-1], hist, width=(edges[1]-edges[0]), alpha=0.7, color=color, label=label)
    # Simple Otsu
    optimal_t = edges[np.argmin(np.abs(np.cumsum(hist)/hist.sum() - 0.5))]
    ax.axvline(x=optimal_t, color='black', lw=2, label=f'Otsu τ*={optimal_t:.4f}')
    ax.set_title(label)
    ax.legend()
plt.tight_layout()
plt.show()


---
## §5 — `extraction/spectral_mask.py`: DiffCut Fiedler Vector

The core spectral partitioning algorithm. Adapts the DiffCut approach (Barsellotti et al., 2023) to FLUX's single-stream attention keys.

In [ ]:
with open('/Users/tella/Workspace/FlowStitch/flowstitch/extraction/spectral_mask.py') as f:
    source = f.read()
print(source)

print("\n" + "─"*60)
print('''
STEP-BY-STEP MATHEMATICAL DERIVATION:

1. INPUT: keys_img ∈ ℝ^{1 × N × D}  (N=4096 sequence tokens, D=feature dim)

2. RESHAPE + DECIMATE (2D bilinear, anti-aliasing):
   K_{2D} ∈ ℝ^{1 × D × 64 × 64}  →  K_down ∈ ℝ^{1 × D × 32 × 32}
   K_flat ∈ ℝ^{1 × 1024 × D}      (1024 = 32×32)
   K_norm = K_flat / ‖K_flat‖_2   (L2 normalize along dim=-1)

3. COSINE AFFINITY MATRIX:
   W = K_norm @ K_norm^T ∈ ℝ^{1024 × 1024}
   W = clamp(W, min=0)   (remove negative affinities)
   diag(W) = 0           (no self-loops)

4. GRAPH LAPLACIAN (UNNORMALIZED):
   D = diag(W·1) ∈ ℝ^{1024 × 1024}  (degree matrix)
   L = D - W                          (combinatorial Laplacian)
   Eigenvalues: 0 = λ₁ ≤ λ₂ ≤ ... ≤ λ_n (by Perron-Frobenius)

5. FIEDLER VECTOR:
   L v = λ v  →  v₂ = Fiedler vector (2nd smallest eigenvalue)
   Computed via: torch.linalg.eigh(L)  (dense symmetric eigensolver)
   The Fiedler vector minimizes the graph cut: min_{v⊥1} v^T L v / ‖v‖²

6. PARTITION (ZERO-CROSSING):
   mask_32 = {1 if v₂[i] > 0 else 0}  (positive half-space)
   
7. UPSAMPLE:
   mask_64 = F.interpolate(mask_32, size=(64,64), mode='nearest')
   mask_seq = mask_64.view(1, 4096, 1)  (back to sequence format)

NOTE: The choice of positive half-space is arbitrary — flip if needed via hybrid_mask.py.
''')


---
## §6 — `extraction/tda_mask.py`: Persistent Homology H₀ Mask

Used when spectral partitioning fails (e.g., near-degenerate graphs). Falls back gracefully to Otsu if `ripser` is not installed.

In [ ]:
with open('/Users/tella/Workspace/FlowStitch/flowstitch/extraction/tda_mask.py') as f:
    source = f.read()
print(source)

print("\n" + "─"*60)
print('''
ALGORITHM OVERVIEW:
  1. Normalize attention mask to [0,1] → apply Otsu → semantic_core (binary)
  2. Extract velocity vectors within semantic core: v_core ∈ ℝ^{|S| × D}
  3. Compute cosine distance matrix: d_ij = 1 - cos(v_i, v_j) ∈ [0, 2]
  4. Run Ripser (optional, for validation): persistent H₀ on cosine distance
  5. Single-linkage clustering: scipy.linkage(condensed_dist, method='single')
  6. Cut at threshold_metric distance → extract largest cluster
  7. Map cluster membership back to full sequence → final binary mask

DESIGN RATIONALE FOR SINGLE-LINKAGE:
  Single-linkage clustering is mathematically equivalent to H₀ persistent homology:
  - Both compute the minimum spanning tree of the distance matrix
  - Both merge components at the same filtration threshold
  - Both identify the largest connected component at threshold τ
  The advantage of single-linkage over Ripser: no external dependency, pure scipy.

FALLBACK CHAIN:
  ripser (full H₀ with persistence diagram)
    → scipy single-linkage (computationally equivalent)
      → otsu_threshold (emergency fallback if clustering fails)
''')


---
## §7 — `extraction/hybrid_mask.py` + §8–§11 — Stitching Stack

In [ ]:
for fname, label in [
    ('/Users/tella/Workspace/FlowStitch/flowstitch/extraction/hybrid_mask.py', 'hybrid_mask.py'),
    ('/Users/tella/Workspace/FlowStitch/flowstitch/stitching/kts.py', 'kts.py'),
    ('/Users/tella/Workspace/FlowStitch/flowstitch/stitching/ema_smoothing.py', 'ema_smoothing.py'),
    ('/Users/tella/Workspace/FlowStitch/flowstitch/stitching/semantic_processor.py', 'semantic_processor.py'),
    ('/Users/tella/Workspace/FlowStitch/flowstitch/stitching/ode_perturbation.py', 'ode_perturbation.py'),
]:
    print(f"\n{'='*60}")
    print(f"── {label} ──")
    print('='*60)
    with open(fname) as f:
        print(f.read())

print('''
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STITCHING STACK MATHEMATICAL SUMMARY:

hybrid_mask.py:
  1. spectral_mask = compute_fiedler_mask(keys_img)  ← structure
  2. semantic_core = otsu_threshold(attn_map)          ← semantics
  3. overlap_check → flip if inverted
  → Final binary mask = correct Fiedler partition

kts.py (Kinetic Trajectory Shaping):
  D(t) = exp(-γ·max(0, t - τ_c))    ← temporal damping
  v_stitch = v_amb + D(t)·λ·M·(v_target - v_amb)
  
  KEY: D(t→1) → 0  prevents terminal singularity as ODE approaches clean image
  KTS guarantees: ‖v_stitch - v_amb‖ → 0 as t → 1

ema_smoothing.py (Look-Back Flows):
  ā_t = γ·a_t + (1-γ)·ā_{t-1}    (exponential moving average)
  Applied to v_stitch between timesteps — reduces jitter from attention oscillations

semantic_processor.py (Semantic Grafting):
  blended = img_out·(1 - α·A) + img_in·(α·A)
  Applied inside EACH attention module of single_transformer_blocks
  → The generated output features are "guided back" to the target features at boundary

ode_perturbation.py (Full ODE Step):
  1. v_ambient = FLUX transformer forward pass (ambient prompt)
  2. v_stitch = apply_kts(v_ambient, v0_target, A_fisica, t_norm)
  3. [optional] v_stitch = EMA(v_stitch)
  4. z_{t+1} = scheduler.step(v_stitch, t, z_t)   (Euler integration)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
''')


---
## §12 — `pipelines/dataset_generation.py` + §13 — `pipelines/latent_stitching.py`

The two main entry points: building the reference database and running the stitching inference.

In [ ]:
for fname, label in [
    ('/Users/tella/Workspace/FlowStitch/flowstitch/pipelines/dataset_generation.py', 'dataset_generation.py'),
    ('/Users/tella/Workspace/FlowStitch/flowstitch/pipelines/latent_stitching.py', 'latent_stitching.py'),
]:
    print(f"\n{'='*65}")
    print(f"── {label} ──")
    print('='*65)
    with open(fname) as f:
        print(f.read())

print('''
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PIPELINE MODES SUMMARY:

  "mosaico" (Hard Quantum Mosaic):
    latents = lake·(1-M) + x0_db·M           ← hard linear blend (DEPRECATED)
    
  "dual" (Variance-Preserving Hybrid):
    A_blurred = gaussian_blur(A_target, σ=2.5)   ← soft boundary
    A_fisica = (A_blurred > 0.1).float()          ← binary physics mask
    latents = lake·√(1-M) + x0_db·√M             ← variance-preserving init
    + semantic_processors injected on single_transformer_blocks
    + KTS + EMA during ODE integration
    
  "full" (Full Dual + Double Blocks):
    Same as "dual" but semantic_processors on BOTH double and single blocks

HISTORICALLY BEST: "dual" mode (as documented in HPC comparative analysis)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
''')


---
## §14 — End-to-End Pipeline Runner

Integration of `run_actual_pipeline.py` — the only end-to-end script that processes real dataset samples through the complete extraction pipeline (attention → Fiedler → KTS → EMA → visualization).

In [ ]:
# ── COMPLETE run_actual_pipeline.py SOURCE (integrated) ─────────────────────
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import sys

sys.path.insert(0, '/Users/tella/Workspace/FlowStitch')

from flowstitch.extraction.spectral_mask import compute_fiedler_mask
from flowstitch.stitching.kts import apply_kts
from flowstitch.stitching.ema_smoothing import AttentionEMA

def process_dataset_sample(sample_dir: str, verbose: bool = True):
    '''
    End-to-end processing of a single dataset sample.
    
    Pipeline:
    1. Load attention_maps.pt and v0_velocity.pt
    2. Extract Fiedler mask via DiffCut spectral matting
    3. Apply KTS damping + EMA smoothing
    4. Save side-by-side visualization
    
    Args:
        sample_dir: Path to dataset sample directory
                   (expects attention_maps.pt, final_image.png, optionally v0_velocity.pt)
    '''
    if verbose:
        print(f"\n{'─'*50}")
        print(f"Sample: {os.path.basename(sample_dir)}")
        print(f"{'─'*50}")
    
    attn_path = os.path.join(sample_dir, "attention_maps.pt")
    v0_path   = os.path.join(sample_dir, "v0_velocity.pt")
    img_path  = os.path.join(sample_dir, "final_image.png")
    
    if not os.path.exists(attn_path) or not os.path.exists(img_path):
        print(f"  ⚠️  Missing files in {sample_dir}, skipping.")
        return None
    
    # ── 1. Load data ─────────────────────────────────────────────────────────
    attn_dict = torch.load(attn_path, map_location="cpu", weights_only=True)
    layer_key = "layer_10" if "layer_10" in attn_dict else list(attn_dict.keys())[0]
    attn_map = attn_dict[layer_key]  # [1, heads, img_tokens, text_tokens]
    if verbose:
        print(f"  Attention map '{layer_key}': {attn_map.shape}")
    
    # ── 2. Spectral Matting (DiffCut) ─────────────────────────────────────────
    # Average across attention heads → [1, N, text_dim]
    attn_features = attn_map.mean(dim=1).to(torch.float32)
    
    # compute_fiedler_mask: decimates to 32×32, computes Laplacian, Fiedler, upsamples
    fiedler_mask = compute_fiedler_mask(attn_features, target_resolution=32)
    # fiedler_mask: [1, 4096, 1]
    mask_2d = fiedler_mask.view(64, 64).numpy()
    if verbose:
        print(f"  Fiedler mask: {fiedler_mask.shape}, active={fiedler_mask.sum().item():.0f}/4096 tokens ({100*fiedler_mask.mean().item():.1f}%)")
    
    # ── 3. KTS + EMA (if v0 available) ────────────────────────────────────────
    results = {'mask_2d': mask_2d, 'layer_key': layer_key}
    
    if os.path.exists(v0_path):
        v0 = torch.load(v0_path, map_location="cpu", weights_only=True).to(torch.float32)
        if verbose:
            print(f"  v0 velocity: {v0.shape}")
        
        # Simulate a target velocity (in production: loaded from reference sample)
        v_target = v0 + torch.randn_like(v0) * 0.05
        
        t_step = 0.90  # Late-time phase (approaching terminal singularity)
        mask_expanded = fiedler_mask.expand_as(v0)
        
        v_damped = apply_kts(
            v_ambient=v0, v_target=v_target, mask=mask_expanded,
            t_norm=t_step, lambda_val=1.0, t_cutoff=0.8, gamma=5.0
        )
        smoother = AttentionEMA(decay=0.3)
        v_smoothed = smoother.update(v_damped)
        
        diff_damped   = (v_damped - v_target).abs().mean().item()
        diff_smoothed = (v_smoothed - v_damped).abs().mean().item()
        if verbose:
            print(f"  KTS diff_damped={diff_damped:.5f}, EMA delta={diff_smoothed:.5f}")
        
        results.update({'v_damped': v_damped, 'v_smoothed': v_smoothed,
                        'diff_damped': diff_damped, 'diff_smoothed': diff_smoothed})
    
    # ── 4. Visualization ───────────────────────────────────────────────────────
    orig_img = Image.open(img_path)
    
    n_cols = 3 if 'v_damped' in results else 2
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))
    
    axes[0].imshow(orig_img)
    axes[0].set_title(f"Generated Image\n({os.path.basename(sample_dir)[:30]})")
    axes[0].axis('off')
    
    im = axes[1].imshow(mask_2d, cmap='viridis', vmin=0, vmax=1)
    axes[1].set_title(f"Fiedler Mask (DiffCut)\n({layer_key}, 32×32→64×64)")
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046)
    
    if 'v_damped' in results:
        # Energy field of KTS-damped velocity
        energy_damped = results['v_damped'].view(64, 64, -1).norm(dim=-1).numpy()
        energy_norm = (energy_damped - energy_damped.min()) / (energy_damped.max() - energy_damped.min() + 1e-8)
        im2 = axes[2].imshow(energy_norm, cmap='hot', vmin=0, vmax=1)
        axes[2].set_title(f"KTS-Damped v₀ Energy\n(t=0.9, diff={results['diff_damped']:.4f})")
        axes[2].axis('off')
        plt.colorbar(im2, ax=axes[2], fraction=0.046)
    
    plt.tight_layout()
    out_path = os.path.join(sample_dir, "pipeline_result.png")
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    if verbose:
        print(f"  ✅ Result saved: {out_path}")
    
    return results


# ── Run on real dataset samples ────────────────────────────────────────────────
DATASET_DIR = '/Users/tella/Workspace/FlowStitch/data/dataset_v1'

samples = []
if os.path.exists(DATASET_DIR):
    for d in sorted(os.listdir(DATASET_DIR)):
        full_path = os.path.join(DATASET_DIR, d)
        if os.path.isdir(full_path) and os.path.exists(os.path.join(full_path, 'attention_maps.pt')):
            samples.append(full_path)

print(f"Found {len(samples)} dataset samples with attention maps:")
for s in samples:
    print(f"  - {os.path.basename(s)}")
print()

if samples:
    # Process first 3 samples
    for sample in samples[:3]:
        try:
            result = process_dataset_sample(sample)
        except Exception as e:
            print(f"  ❌ Error on {os.path.basename(sample)}: {e}")
else:
    print("⚠️  No samples found with attention_maps.pt.")
    print("   Run flowstitch/pipelines/dataset_generation.py first to build the dataset.")
    print("   Required structure: data/dataset_v1/<prompt_slug>/{attention_maps.pt, final_image.png, v0_velocity.pt}")


---
## §15 — Notebook Generator Reference

For reproducibility, the generator scripts that built notebooks 01–05 and the Master Walkthrough are preserved here.

In [ ]:
# ── Generator scripts location ───────────────────────────────────────────────
import os

generators = {
    'generate_individual_notebooks.py': 'Generates notebooks 01-05 (theory + experiments)',
    'create_master_notebook.py': 'Generates Master_Thesis_Walkthrough.ipynb',
}

for script, desc in generators.items():
    script_path = os.path.join(
        '/Users/tella/Workspace/FlowStitch/local_analysis', script
    )
    size_kb = os.path.getsize(script_path) / 1024 if os.path.exists(script_path) else 0
    print(f"{'─'*55}")
    print(f"Script: {script}  ({size_kb:.1f} KB)")
    print(f"Purpose: {desc}")
    print(f"Location: {script_path}")
    print(f"Exists: {os.path.exists(script_path)}")
    print(f'''
To regenerate notebooks:
  cd /Users/tella/Workspace/FlowStitch
  python local_analysis/{script}
''')
print("─"*55)
print('''
IMPORTANT: These generator scripts should be re-run any time you want to
regenerate notebooks 01-05 or the Master Walkthrough from scratch.
They use nbformat to programmatically build structured research notebooks.
''')
